# 03 · Stability tests

A linear system is asymptotically stable when every pole has a negative real part (every eigenvalue of A).
Before computers one had to decide that without finding the roots, and the classical tests still matter: the
Routh array and Hurwitz determinants decide stability from the polynomial's coefficients, give the number of
unstable roots, and - most usefully - give stability as a condition on a parameter such as a gain. Lyapunov's
method decides it from a quadratic form and extends to nonlinear systems. For discrete systems Jury's test does
the same inside the unit circle. This notebook runs each test and checks it against the roots.

**What you will learn**
- build the Routh array including its special cases and count unstable roots
- use Hurwitz determinants and Lyapunov's equation
- derive stability conditions on parameters
- apply Jury's test to discrete systems and recognise the conditioning problem of polynomial coefficients

**Before you start:** notebook 00  ·  **Time:** about 60 min

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, stability as st, polynomials as pl, linalg as la, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## The Routh array

In [2]:
for c in ([1, 2, 3, 4], [1, 1, 2, 2, 3], [1, 2, 8, 12, 20, 16, 16]):
    r = st.routh(c)
    print(f"polynomial {c}:\n{np.round(r['table'], 4)}\n  sign changes in the first column: {r['rhp_roots']}   actual right-half-plane roots: {int(np.sum(pl.roots(c).real > 1e-9))}" + (f"   auxiliary polynomial {r['auxiliary']} (roots {np.round(pl.roots(r['auxiliary']), 3)})" if r["auxiliary"] is not None else "") + "\n")

polynomial [1, 2, 3, 4]:
[[1. 3.]
 [2. 4.]
 [1. 0.]
 [4. 0.]]
  sign changes in the first column: 0   actual right-half-plane roots: 0

polynomial [1, 1, 2, 2, 3]:
[[ 1.e+00  2.e+00  3.e+00]
 [ 1.e+00  2.e+00  0.e+00]
 [ 0.e+00  3.e+00  0.e+00]
 [-3.e+09  0.e+00  0.e+00]
 [ 3.e+00  0.e+00  0.e+00]]
  sign changes in the first column: 2   actual right-half-plane roots: 2

polynomial [1, 2, 8, 12, 20, 16, 16]:
[[ 1.      8.     20.     16.    ]
 [ 2.     12.     16.      0.    ]
 [ 2.     12.     16.      0.    ]
 [ 8.     24.      0.      0.    ]
 [ 6.     16.      0.      0.    ]
 [ 2.6667  0.      0.      0.    ]
 [16.      0.      0.      0.    ]]
  sign changes in the first column: 0   actual right-half-plane roots: 0   auxiliary polynomial [ 2.  0. 12.  0. 16.] (roots [-0.+2.j    -0.-2.j     0.+1.414j  0.-1.414j])



The array's first column changes sign once for each root in the right half-plane. Two special cases need
care: a zero in the first column (the ε method replaces it by a small number and the signs are read in the
limit), and a whole row of zeros, which signals roots symmetric about the origin - on the imaginary axis or in
± pairs - and is handled by differentiating the auxiliary polynomial formed from the row above; its roots
(±j√2, ±2j in the last example) are the marginal ones.

## Stability as a condition on a gain

In [3]:
K = 1.0
G = sy.tf([1.0], pl.from_roots([0, -1, -2]))                       # 1 / (s (s + 1)(s + 2))
print("closed-loop polynomial s^3 + 3 s^2 + 2 s + K: Routh's condition is 3*2 > K, i.e. K < 6")
for K in (1.0, 5.9, 6.0, 6.1):
    r = st.routh(pl.add(G.den, K * G.num))
    print(f"   K = {K}: first column {np.round(r['table'][:, 0], 3)} -> {'stable' if r['stable'] else 'NOT stable'}" + (f" (auxiliary: roots at {np.round(pl.roots(r['auxiliary']), 3)})" if r["auxiliary"] is not None else ""))

closed-loop polynomial s^3 + 3 s^2 + 2 s + K: Routh's condition is 3*2 > K, i.e. K < 6
   K = 1.0: first column [1.    3.    1.667 1.   ] -> stable
   K = 5.9: first column [1.    3.    0.033 5.9  ] -> stable
   K = 6.0: first column [1. 3. 6. 6.] -> NOT stable (auxiliary: roots at [-0.+1.414j  0.-1.414j])
   K = 6.1: first column [ 1.     3.    -0.033  6.1  ] -> NOT stable


Here the array decides more than stability: it gives the range of K for which the loop is stable (K < 6) and
the frequency at which it goes marginal (the auxiliary polynomial 3s² + 6 at K = 6 has roots ±j√2). Both
numbers reappear in the root locus (notebook 04) and in the gain margin (notebook 02): three views, one fact.

## Hurwitz, Lyapunov, Jury

In [4]:
c = [1, 2, 3, 4]
hz = st.hurwitz(c)
print("Hurwitz matrix:\n", hz["matrix"], "\nleading minors:", np.round(hz["minors"], 3), "-> stable:", hz["stable"])
A = models.mass_spring_damper()["system"].A
ly = st.lyapunov_test(A)
print("\nLyapunov: A^T P + P A = -I gives P =\n", np.round(ly["P"], 4), "\neigenvalues of P", np.round(ly["eigenvalues"], 4), "-> positive definite:", ly["stable"])
print("verification: A^T P + P A + I =", np.max(np.abs(A.T @ ly["P"] + ly["P"] @ A + np.eye(2))))
for cz in ([1, -1.5, 0.56], [1, 0.5, -0.5], [1, -0.3, 0.2, -0.1]):
    j = st.jury(cz)
    print(f"Jury for {cz}: conditions {['ok' if ok else 'FAIL' for ok in j['conditions']]} -> stable {j['stable']}; root moduli {np.round(np.abs(pl.roots(cz)), 3)}")

Hurwitz matrix:
 [[2. 4. 0.]
 [1. 3. 0.]
 [0. 2. 4.]] 
leading minors: [2. 2. 8.] -> stable: True

Lyapunov: A^T P + P A = -I gives P =
 [[6.3    0.125 ]
 [0.125  1.5625]] 
eigenvalues of P [1.5592 6.3033] -> positive definite: True
verification: A^T P + P A + I = 0.0
Jury for [1, -1.5, 0.56]: conditions ['ok', 'ok', 'ok'] -> stable True; root moduli [0.8 0.7]
Jury for [1, 0.5, -0.5]: conditions ['ok', 'FAIL', 'ok'] -> stable False; root moduli [1.  0.5]
Jury for [1, -0.3, 0.2, -0.1]: conditions ['ok', 'ok', 'ok', 'ok'] -> stable True; root moduli [0.495 0.495 0.409]


Hurwitz's criterion is the Routh array in determinant form: all leading principal minors of the Hurwitz matrix
positive. Lyapunov's is different in kind: if a positive-definite quadratic form V = xᵀPx decreases along every
trajectory (AᵀP + PA negative definite), the origin is stable - and for a linear system the equation
AᵀP + PA = −Q always has a positive-definite solution exactly when A is stable. The library solves it by
Bartels-Stewart on the real Schur form (notebook 18). Jury's test is the discrete-time counterpart: conditions
on the coefficients that place all roots inside the unit circle, which the moduli of the roots confirm.

## When the tests disagree with the roots: conditioning

In [5]:
roots_true = -np.arange(1, 13, dtype=float)                   # 12 well-separated stable roots
c = pl.from_roots(roots_true)
print(f"coefficients span {c.max():.2e} to {c.min():.2e}; Routh says {st.routh(c)['rhp_roots']} RHP roots, roots computed from the coefficients: {np.round(np.sort(pl.roots(c).real), 3)}")
c_pert = c * (1 + 1e-8 * np.array([1, -1] * 6 + [1]))         # relative perturbation of 1e-8 in the coefficients
print(f"after perturbing the coefficients by 1e-8: roots {np.round(np.sort(pl.roots(c_pert).real), 3)}")

coefficients span 1.93e+09 to 1.00e+00; Routh says 0 RHP roots, roots computed from the coefficients: [-12. -11. -10.  -9.  -8.  -7.  -6.  -5.  -4.  -3.  -2.  -1.]
after perturbing the coefficients by 1e-8: roots [-11.785 -11.785  -9.637  -9.637  -7.275  -7.275  -5.319  -5.319  -3.966
  -3.003  -2.     -1.   ]


Twelve roots at −1 … −12 give coefficients from 1 to 4.8×10⁸, and a relative perturbation of 10⁻⁸ in them -
the size of a measurement's rounding - moves the roots by tenths. Wilkinson's famous example in miniature:
polynomial coefficients are a poorly conditioned way to carry a high-order system, which is why the
state-space representation and eigenvalue computations on A are preferred numerically (notebook 18), and why
the classical tests are at their best for low-order systems and symbolic conditions.

## Inside the algorithm: one Routh row by hand

In [6]:
c = [1, 2, 3, 4]
r1, r2 = np.array([c[0], c[2]], float), np.array([c[1], c[3]], float)
r3 = np.array([(r2[0] * r1[1] - r1[0] * r2[1]) / r2[0], 0.0])
r4 = np.array([(r3[0] * r2[1] - r2[0] * r3[1]) / r3[0], 0.0])
print("rows by hand:", r1, r2, r3, r4, "\nlibrary table:\n", st.routh(c)["table"])

rows by hand: [1. 3.] [2. 4.] [1. 0.] [4. 0.] 
library table:
 [[1. 3.]
 [2. 4.]
 [1. 0.]
 [4. 0.]]


## Exercises
1. The loop 1 + K (s + a)/(s^2 (s + 10)) = 0: use the Routh array symbolically (write the conditions on K and a) and confirm numerically for a = 1 and a = 5 the range of K for stability.
2. For the levitated ball (`models.magnetic_levitation`) with proportional feedback u = -K x, show with the Routh array that no K stabilises it, and that u = -K x - Kd x' does for suitable Kd. Confirm with the eigenvalues.
3. **Implement it yourself:** solve the Lyapunov equation A^T P + P A = -Q for a 3 x 3 A by the Kronecker product (vec form) and compare with `linalg.lyapunov`; then show that P fails to be positive definite when A has an unstable eigenvalue.